In [ ]:
# import necessary libs and util functions and save intermediate results for validation and the final dataframe with the mask 
# and reflectance data and also numpy files for mask


In [ ]:
# 1. Learn what are .bin and .hdr file formats and how they store the data of the reflectance data
# 2. Extract the reflectance data from these files
# 3. Train an SVM classifier and extract the mask, save the mask, and extract the reflectance data - save the mask, reflectance data.
# 4. Save the reflectance for the each sample separately.
# 5. Finally aggregate the average reflectance for each sample into the final CSV file. 



In [ ]:
%matplotlib qt
import numpy as np

from skimage.morphology import remove_small_objects, remove_small_holes, opening, disk
import spectral.io.envi as envi
import matplotlib.pyplot as plt
from pathlib import Path
import pandas as pd
import numpy as np
import os

### Reflectance - image cube

In [ ]:
import spectral.io.envi as envi

file_placeholder = r"C:\Users\baolab\Downloads\biochar_data\control"

header_file = f'{file_placeholder}/Control 16_27_2026-02-26_11-05-42_.hdr'
spectral_file = f'{file_placeholder}/Control 16_27_2026-02-26_11-05-42_.bin'

image_cube = envi.open(header_file, spectral_file)

In [ ]:
image_cube

	Data Source:   'C:\Users\baolab\Downloads\biochar_data\control/Control 16_27_2026-02-26_11-05-42_.bin'
	# Rows:            630
	# Samples:         640
	# Bands:           201
	Interleave:        BIP
	Quantization:  32 bits
	Data format:   float32

In [ ]:
hypercube_arr = np.array(image_cube.load())


c:\ProgramData\anaconda3\envs\hyperspec\Lib\site-packages\spectral\io\spyfile.py:224: NaNValueWarning: Image data contains NaN values.
  warnings.warn('Image data contains NaN values.', NaNValueWarning)


In [ ]:
import matplotlib.pyplot as plt

def get_plant_px(hypercube):

    display_band = hypercube[:, :, 50]

    plant_coords = [] # List to store plant coordinates

    fig, ax = plt.subplots()

    ax.imshow(display_band, cmap='viridis')
    ax.set_title("Click to select PLANT pixels")

    def onclick_plant(event):

        # Get integer coordinates of the click
        x, y = int(event.xdata), int(event.ydata)

        # Store the coordinates (row, col)
        plant_coords.append([y, x])

        # Draw a marker for visual feedback
        ax.plot(x, y, 'rx', markersize=8)
        fig.canvas.draw()

    # Connect the click event to our function
    cid = fig.canvas.mpl_connect('button_press_event', onclick_plant)
    plt.show() # Display the interactive plot

    return plant_coords

def get_background_px(hypercube):

    display_band = hypercube[:, :, 50]

    background_coords = [] # List for background coordinates
    fig2, ax2 = plt.subplots()
    ax2.imshow(display_band, cmap='viridis')
    ax2.set_title("Click to select BACKGROUND pixels")

    def onclick_background(event):
        x, y = int(event.xdata), int(event.ydata)
        background_coords.append([y, x])
        ax2.plot(x, y, 'bo', markersize=5, markerfacecolor='none') # Blue circle marker
        fig2.canvas.draw()

    cid2 = fig2.canvas.mpl_connect('button_press_event', onclick_background)
    plt.show()

    return background_coords


In [10]:
plantsp = get_plant_px(image_cube)

In [12]:
print(plantsp)


[[271, 246], [268, 270], [266, 278], [265, 283], [267, 289], [268, 295], [271, 298], [279, 295], [287, 295], [290, 297], [296, 301], [309, 301], [310, 286], [297, 287], [287, 284], [282, 284], [276, 272], [276, 267], [277, 237], [287, 243], [295, 240], [302, 233], [328, 247], [341, 253], [345, 269], [345, 269], [341, 275], [338, 282], [334, 288], [330, 292], [318, 285], [321, 275], [321, 275], [323, 269], [323, 269], [325, 265], [304, 260], [304, 260], [305, 262], [307, 267], [303, 271], [301, 272], [295, 275], [291, 271], [289, 259], [289, 259], [286, 256], [280, 257], [277, 256], [270, 256], [272, 263], [271, 274], [276, 287], [281, 274], [310, 234], [310, 224], [314, 224], [320, 230], [316, 240], [333, 243], [323, 243], [329, 268], [335, 259], [317, 255], [308, 246], [305, 246], [297, 250], [284, 235], [261, 240], [264, 244], [270, 258], [276, 260], [285, 262], [318, 295], [308, 295], [300, 295], [303, 302], [334, 302], [326, 302], [328, 297], [325, 287], [331, 276], [333, 272], [34

In [ ]:
plantsp = [[271, 246], [268, 270], [266, 278], [265, 283], [267, 289], [268, 295], [271, 298], [279, 295], [287, 295], [290, 297], [296, 301], [309, 301], [310, 286], [297, 287], [287, 284], [282, 284], [276, 272], [276, 267], [277, 237], [287, 243], [295, 240], [302, 233], [328, 247], [341, 253], [345, 269], [345, 269], [341, 275], [338, 282], [334, 288], [330, 292], [318, 285], [321, 275], [321, 275], [323, 269], [323, 269], [325, 265], [304, 260], [304, 260], [305, 262], [307, 267], [303, 271], [301, 272], [295, 275], [291, 271], [289, 259], [289, 259], [286, 256], [280, 257], [277, 256], [270, 256], [272, 263], [271, 274], [276, 287], [281, 274], [310, 234], [310, 224], [314, 224], [320, 230], [316, 240], [333, 243], [323, 243], [329, 268], [335, 259], [317, 255], [308, 246], [305, 246], [297, 250], [284, 235], [261, 240], [264, 244], [270, 258], [276, 260], [285, 262], [318, 295], [308, 295], [300, 295], [303, 302], [334, 302], [326, 302], [328, 297], [325, 287], [331, 276], [333, 272], [343, 260], [325, 255], [315, 265], [313, 272], [306, 280], [315, 231], [295, 281]]
print(plantsp)

In [66]:

backpx = get_background_px(image_cube)


In [67]:

print(backpx)

[[700, 23], [696, 100], [694, 131], [698, 159], [695, 196], [674, 139], [686, 188], [678, 170], [669, 210], [668, 241], [667, 261], [663, 295], [674, 345], [659, 383], [669, 401], [675, 422], [646, 530], [684, 618], [618, 618], [559, 604], [520, 610], [469, 610], [439, 610], [398, 614], [355, 614], [306, 614], [268, 614], [241, 613], [209, 613], [160, 610], [116, 612], [87, 612], [61, 611], [21, 607], [20, 593], [26, 579], [33, 587], [33, 591], [33, 592], [40, 623], [48, 604], [54, 593], [53, 549], [50, 509], [36, 471], [40, 481], [40, 494], [40, 512], [40, 529], [44, 574], [51, 612], [15, 426], [14, 446], [10, 490], [10, 513], [10, 553], [15, 630], [16, 309], [36, 279], [40, 249], [42, 263], [41, 282], [45, 306], [41, 322], [55, 312], [61, 271], [56, 250], [57, 276], [61, 314], [57, 317], [48, 445], [39, 465], [51, 474], [55, 499], [40, 12], [42, 27], [42, 54], [43, 80], [43, 90], [43, 104], [9, 27], [9, 78], [8, 141], [17, 191], [6, 236], [6, 271], [14, 366], [55, 379], [84, 10], [67

In [68]:
hypercube_arr.shape

(720, 640, 201)

In [47]:
np.nanmean(hypercube_arr[:, :, 0])

0.3908703

In [51]:
np.nanmean(hypercube_arr[:, :, 200])

0.076726936

In [52]:
np.nanmean(hypercube_arr[:, :, 120])

0.19843492

In [56]:
new_cube = np.nan_to_num(hypercube_arr, nan=0.0)
new_cube

array([[[0.53088236, 0.475     , 0.48783782, ..., 0.11875   ,
         0.11014493, 0.11026785],
        [0.5029412 , 0.5277778 , 0.53918916, ..., 0.1180791 ,
         0.10935251, 0.11227273],
        [0.5135135 , 0.5277778 , 0.52500004, ..., 0.12344632,
         0.11014493, 0.12954545],
        ...,
        [0.5181818 , 0.58055556, 0.5428571 , ..., 0.1443038 ,
         0.1368    , 0.133     ],
        [0.5588235 , 0.53088236, 0.53088236, ..., 0.14339623,
         0.1281746 , 0.13168317],
        [0.6333333 , 0.57575756, 0.57      , ..., 0.14188312,
         0.1292    , 0.11399999]],

       [[0.53088236, 0.4486111 , 0.53918916, ..., 0.13494319,
         0.11702899, 0.1017857 ],
        [0.5588235 , 0.5013889 , 0.5905405 , ..., 0.12881356,
         0.11618704, 0.12090909],
        [0.5905405 , 0.5277778 , 0.55      , ..., 0.13418078,
         0.11014493, 0.11227273],
        ...,
        [0.5469697 , 0.5277778 , 0.5428571 , ..., 0.13829114,
         0.1292    , 0.133     ],
        [0.5

In [172]:

def train_svm(image_cube, plant_coords, background_coords):

    # hypercube = image_cube.load()
    hypercube = image_cube
    # --- Build the training data from your selected points ---
    X_train = []
    y_train = []
    bands = hypercube.shape[2]

    # Add plant pixels (Class 1)
    for r, c in plant_coords:
        X_train.append(hypercube[r, c, :].reshape(bands))
        y_train.append(1)

    # Add background pixels (Class 0)
    for r, c in background_coords:
        X_train.append(hypercube[r, c, :].reshape(bands))
        y_train.append(0)

    X_train = np.array(X_train)
    y_train = np.array(y_train)

    print(f"Training data created successfully!")
    print(f"Plant samples: {len(plant_coords)}")
    print(f"Background samples: {len(background_coords)}")
    print(f"Total training samples: {X_train.shape[0]}")

    # --- Step 2: Train the SVM Classifier ---
    # The 'C' and 'gamma' parameters can be tuned, but defaults are a good start.
    from sklearn.svm import SVC

    classifier = SVC(C=550, gamma='scale')
    classifier.fit(X_train, y_train)
    print("Classifier trained successfully!")

    # --- Step 3: Predict on the Entire Image ---
    # Reshape the hypercube to a long list of pixels (n_pixels, n_bands)
    rows, cols, bands = hypercube.shape
    img_reshaped = hypercube.reshape(rows * cols, bands)

    # Predict the class for every pixel
    predicted_labels = classifier.predict(img_reshaped)

    # Reshape the prediction back into a 2D image mask
    final_mask = predicted_labels.reshape(rows, cols)

    # --- Visualize the Result ---
    plt.figure(figsize=(8, 8))
    plt.imshow(final_mask, cmap='gray')
    plt.title('Segmentation Mask from SVM Classifier')
    plt.show()

    return final_mask, classifier

def predict_svm(svm_classifier, hypercube):
    
    rows, cols, bands = hypercube.shape
    img_reshaped = hypercube.reshape(rows * cols, bands)

    # Predict the class for every pixel
    predicted_labels = svm_classifier.predict(img_reshaped)

    # Reshape the prediction back into a 2D image mask
    final_mask = predicted_labels.reshape(rows, cols)

    return final_mask


In [70]:

svm_mask, svm_classifier = train_svm(new_cube, plantsp, backpx)


Training data created successfully!
Plant samples: 115
Background samples: 342
Total training samples: 457
Classifier trained successfully!


In [72]:
single_band = image_cube.read_band(80)

In [74]:
fig, axes = plt.subplots(1, 2, figsize=(12, 6))

axes[0].imshow(svm_mask, cmap='gray')
axes[0].set_title('SVM')

axes[1].imshow(single_band, cmap='gray')
axes[1].set_title('Final Mask New')

Text(0.5, 1.0, 'Final Mask New')

In [90]:
# --- Visualize the huge improvement ---

def viz(final_mask, img, filename=None, random_band=45):

    fig, axes = plt.subplots(1, 2, figsize=(9, 5))

    single_band = img.read_band(random_band)
    axes[0].imshow(single_band, cmap='gray')
    axes[0].set_title('Origninal')

    axes[1].imshow(final_mask, cmap='gray')
    axes[1].set_title('SVM Mask')

    # plt.savefig(f'results/reflectance{ref_num}/images/{filename}.png', dpi=300)

    plt.show()


In [86]:
# save bool-mask as numpy file 
import os
import numpy as np
import pandas as pd

def avg_reflectance(img, mask, filename, ref_num):
    hypercube_arr = np.array(img.load())
    mask_bool = mask.astype(bool)

    # Extract plant pixels from the boolean mask
    plant_pixels = hypercube_arr[mask_bool, :]

    # Calculate the average reflectance across all plant pixels for each band
    average_reflectance = np.mean(plant_pixels, axis=0)

    print(f'Total Plant pixels: {plant_pixels.shape[0]}')

    # Create required directories if they don't exist
    csv_dir = f'results/reflectance{ref_num}/csv'
    numpy_dir = f'results/reflectance{ref_num}/numpy_masks'
    os.makedirs(csv_dir, exist_ok=True)
    os.makedirs(numpy_dir, exist_ok=True)

    # Save the average reflectance into a CSV file
    data = {
        'wavelengths': img.bands.centers,
        'reflectance': average_reflectance
    }

    df = pd.DataFrame(data)
    df.to_csv(os.path.join(csv_dir, f'{filename}.csv'), index=False)

    # Save the numpy array mask
    np.save(os.path.join(numpy_dir, f'{filename}.npy'), mask_bool)


In [87]:
# --- Visualize ---

def save_mask(mask, filename=None):

    fig, axes = plt.subplots(1, 1, figsize=(4, 7))
    plt.tight_layout()
    axes.imshow(mask, cmap='gray')
    
    # file_path = f'results/reflectance{ref_num}/masks/'
    
    # if not os.path.exists(file_path):
    #     os.makedirs(file_path, exist_ok=True)
    
    # plt.savefig(f'{file_path}/{filename}.png', dpi=300, bbox_inches='tight')
    plt.close(fig)
    
def viz_all(initial_mask, final_mask, img, filename=None, random_band=45, is_mask=False):

    fig, axes = plt.subplots(1, 3, figsize=(9, 5))

    axes[0].imshow(initial_mask, cmap='gray')
    axes[0].set_title('Initial Mask')

    single_band = img.read_band(random_band)

    axes[1].imshow(single_band, cmap='gray')
    axes[1].set_title('Origninal')

    axes[2].imshow(final_mask, cmap='gray')
    axes[2].set_title('Final Mask')

    # comp_file_path = f'results/reflectance{ref_num}/masks/compare/'
    # if not os.path.exists(comp_file_path):
    #     os.makedirs(comp_file_path, exist_ok=True)

    # plt.savefig(f'{comp_file_path}/{filename}.png', dpi=300, bbox_inches='tight')

    plt.show()

    if is_mask:
        save_mask(final_mask, filename)
    # plt.close()

In [ ]:

# --- Post-processing steps ---

def post_process(mask, max_size_small_objects=1200, max_size_holes=1, radius=3.5):

    # 1. Remove small white objects (noise in the background)
    # '150' is the minimum size in pixels for an object to be kept. You can adjust this.

    # converting the integer array with 0 and 1s into a boolean as skimage.morhology expects boolean array
    mask = mask.astype(bool)
    mask1 = remove_small_objects(mask, max_size=max_size_small_objects)

    # 2. Fill small black holes inside the plant
    # '150' is the maximum size of a hole to be filled.
    mask2 = remove_small_holes(mask1, max_size=max_size_holes)
    
    # 3. Opening: 
    selem = disk(radius)

    # Apply the opening operation
    final_mask = opening(mask2, selem)

    return final_mask

In [96]:
final_mask = post_process(svm_mask, 300, 2, 1)
viz_all(svm_mask, final_mask, image_cube)


In [ ]:
is_filter = True

if is_filter:
    final_mask = post_process(svm_mask, 877, 1, 1.5)
    # visualize the masks befire and after the noise filtering 
    viz_all(svm_mask, final_mask, new_cube)
else:
    viz(final_mask=svm_mask, img=new_cube)


In [98]:

mask_uint8 = (final_mask * 255).astype(np.uint8)


In [100]:
from PIL import Image
mask_path = 'binary_mask_svm.png'
Image.fromarray(mask_uint8).save(mask_path)

In [139]:
bands_list = image_cube.bands.centers

print(bands_list)

[900.0, 904.0, 908.0, 912.0, 916.0, 920.0, 924.0, 928.0, 932.0, 936.0, 940.0, 944.0, 948.0, 952.0, 956.0, 960.0, 964.0, 968.0, 972.0, 976.0, 980.0, 984.0, 988.0, 992.0, 996.0, 1000.0, 1004.0, 1008.0, 1012.0, 1016.0, 1020.0, 1024.0, 1028.0, 1032.0, 1036.0, 1040.0, 1044.0, 1048.0, 1052.0, 1056.0, 1060.0, 1064.0, 1068.0, 1072.0, 1076.0, 1080.0, 1084.0, 1088.0, 1092.0, 1096.0, 1100.0, 1104.0, 1108.0, 1112.0, 1116.0, 1120.0, 1124.0, 1128.0, 1132.0, 1136.0, 1140.0, 1144.0, 1148.0, 1152.0, 1156.0, 1160.0, 1164.0, 1168.0, 1172.0, 1176.0, 1180.0, 1184.0, 1188.0, 1192.0, 1196.0, 1200.0, 1204.0, 1208.0, 1212.0, 1216.0, 1220.0, 1224.0, 1228.0, 1232.0, 1236.0, 1240.0, 1244.0, 1248.0, 1252.0, 1256.0, 1260.0, 1264.0, 1268.0, 1272.0, 1276.0, 1280.0, 1284.0, 1288.0, 1292.0, 1296.0, 1300.0, 1304.0, 1308.0, 1312.0, 1316.0, 1320.0, 1324.0, 1328.0, 1332.0, 1336.0, 1340.0, 1344.0, 1348.0, 1352.0, 1356.0, 1360.0, 1364.0, 1368.0, 1372.0, 1376.0, 1380.0, 1384.0, 1388.0, 1392.0, 1396.0, 1400.0, 1404.0, 1408.0, 

In [101]:
# %matplotlib qt
import numpy as np
from pathlib import Path
from utils.utils import run
from utils.train import SVMclassifier
from lima_beans.bean_utils import QtLinePicker, process_and_save_objects

In [143]:
import os
import numpy as np
import pandas as pd
from skimage.measure import label, regionprops
from PIL import Image

def get_biochar_reflectance(hypercube, mask, output_dir="processed_objects"):
    """
    Saves the mask, extracts average reflectance per object into a DataFrame 
    using wavelength columns (900 to 1700, step 4), 
    and saves isolated .npy cubes with np.nan backgrounds.
    """
    os.makedirs(output_dir, exist_ok=True)

    # Save the overall 2D mask
    np.save(os.path.join(output_dir, "full_mask.npy"), mask)
    print("Saved full mask.")

    labeled_mask = label(mask)
    regions = regionprops(labeled_mask)
    
    print(f"Found {len(regions)} distinct objects. Processing...")

    # Generate the wavelength array: 900 to 1700 (inclusive) with a step of 4
    # This creates exactly 201 bands: [900, 904, 908, ..., 1700]
    wavelengths = np.arange(900, 1704, 4)

    # List to hold the dictionary rows for our Pandas DataFrame
    all_object_averages = []

    for i, region in enumerate(regions):
        object_id = f"object_{i+1:02d}"
        
        # Get the bounding box and crop
        min_y, min_x, max_y, max_x = region.bbox
        cropped_cube = hypercube[min_y:max_y, min_x:max_x, :]
        
        # Create a boolean mask for JUST this object within its bounding box
        cropped_mask = labeled_mask[min_y:max_y, min_x:max_x] == (i + 1)
        
        # Extract ONLY the True pixels
        true_object_pixels = cropped_cube[cropped_mask] 
        
        # Calculate the average reflectance for each band
        mean_reflectance = np.mean(true_object_pixels, axis=0)
        
        # Build the row dictionary: ID, then Wavelengths
        row_data = {'Object_ID': object_id}
        for band_idx, refl_val in enumerate(mean_reflectance):
            # Map the calculated average to the specific wavelength column
            if band_idx < len(wavelengths):
                row_data[f'{wavelengths[band_idx]}'] = refl_val
            
        all_object_averages.append(row_data)
        
        # Eliminating Bounding Box Background
        # Cast to float so we can use np.nan for the background pixels
        cropped_cube_isolated = cropped_cube.astype(float) 
        cropped_cube_isolated[~cropped_mask] = np.nan 
        
        # Save the individual numpy file
        mask_path = f"{object_id}.png"
        Image.fromarray(mask_uint8).save(os.path.join(output_dir, mask_path))
        # np.save(os.path.join(output_dir, filename), cropped_cube_isolated)
        
    # Build the final CSV with rows = objects, columns = wavelengths
    df_averages = pd.DataFrame(all_object_averages)
    df_averages.to_csv(os.path.join(output_dir, "per_object_reflectance.csv"), index=False)
    
    print(f"Successfully saved {len(regions)} individual cubes and the reflectance CSV.")

In [144]:
get_biochar_reflectance(new_cube, final_mask, output_dir="sample_biochar2")

Saved full mask.
Found 1 distinct objects. Processing...
Successfully saved 1 individual cubes and the reflectance CSV.


In [2]:
import pandas as pd
import matplotlib.pyplot as plt

In [3]:
dfx1 = pd.read_csv(r'processed_objectsX1\per_object_reflectance.csv')
dfx1

,Object_ID,900,904,908,912,916,920,924,928,932,...,1664,1668,1672,1676,1680,1684,1688,1692,1696,1700
0,object_01,0.099333,0.095914,0.095725,0.095604,0.097353,0.096459,0.097653,0.096678,0.096837,...,0.100501,0.099933,0.099948,0.098863,0.098019,0.096746,0.095831,0.095173,0.095396,0.096034


In [164]:
import joblib

# Assuming 'trained_classifier' is your trained sklearn SVM object
joblib.dump(svm_classifier, 'trained_svm_model.joblib')


['trained_svm_model.joblib']

In [176]:
new_maskx = predict_svm(svm_classifier, new_cube)
viz(new_maskx, image_cube)

In [ ]:
predict_svm

In [167]:
import os
import numpy as np
import pandas as pd
import joblib
from skimage import io, img_as_ubyte
from skimage.measure import label, regionprops

def predict_and_extract(hypercube, model_path, output_dir="processed_predictions"):
    os.makedirs(output_dir, exist_ok=True)
    
    # 1. Load the model and new hyperspectral cube
    svm_model = joblib.load(model_path)
    # hypercube = np.load(hypercube_path)
    height, width, bands = hypercube.shape
    
    # 2. Predict the mask
    flattened_pixels = hypercube.reshape(-1, bands)
    predictions = svm_model.predict(flattened_pixels)
    binary_mask = predictions.reshape(height, width).astype(bool)
    
    # 3. Save the binary mask as a PNG
    # Convert boolean mask to 0 and 255 uint8 format for PNG saving
    png_mask = img_as_ubyte(binary_mask)
    io.imsave(os.path.join(output_dir, "predicted_mask.png"), png_mask)
    
    # 4. Process distinct objects
    labeled_mask = label(binary_mask)
    regions = regionprops(labeled_mask)
    
    wavelengths = np.arange(900, 1704, 4)
    all_object_averages = []
    
    for i, region in enumerate(regions):
        object_id = f"object_{i+1:02d}"
        
        # Crop to bounding box
        min_y, min_x, max_y, max_x = region.bbox
        cropped_cube = hypercube[min_y:max_y, min_x:max_x, :]
        cropped_mask = labeled_mask[min_y:max_y, min_x:max_x] == (i + 1)
        
        # Calculate average reflectance
        true_object_pixels = cropped_cube[cropped_mask] 
        mean_reflectance = np.mean(true_object_pixels, axis=0)
        
        row_data = {'Object_ID': object_id}
        for band_idx, refl_val in enumerate(mean_reflectance):
            if band_idx < len(wavelengths):
                row_data[f'{wavelengths[band_idx]}'] = refl_val
                
        all_object_averages.append(row_data)
        
        # Isolate object and pad background with NaN
        cropped_cube_isolated = cropped_cube.astype(float) 
        cropped_cube_isolated[~cropped_mask] = np.nan 
        
        # Save isolated NumPy cube
        np.save(os.path.join(output_dir, f"{object_id}.npy"), cropped_cube_isolated)
        
    # 5. Save the CSV
    df_averages = pd.DataFrame(all_object_averages)
    df_averages.to_csv(os.path.join(output_dir, "predicted_reflectance.csv"), index=False)


In [ ]:

hypercube_cube = new_cube
trained_model_file = "trained_svm_model.joblib"

predict_and_extract(hypercube_cube, trained_model_file, output_dir='sample_biochar3')

In [146]:
dfx1 = dfx1.T
dfx1

,0
Object_ID,object_01
900,0.108301
904,0.104816
908,0.105945
912,0.105886
...,...
1684,0.072075
1688,0.070623
1692,0.069486
1696,0.070124


In [159]:
refs = dfx1[0].to_list()
print(refs[1:])

[0.1083006, 0.10481555, 0.10594453, 0.1058864, 0.10709565, 0.10674604, 0.10435827, 0.10109467, 0.10029104, 0.09284026, 0.08328641, 0.07631554, 0.07269318, 0.07033518, 0.06946219, 0.06821746, 0.067814305, 0.067465685, 0.06722218, 0.06705635, 0.06696933, 0.06694226, 0.06694638, 0.06692712, 0.06683836, 0.06670732, 0.06663335, 0.066416286, 0.06645757, 0.06611998, 0.06616888, 0.0662657, 0.0663398, 0.06630228, 0.066296935, 0.0663949, 0.06640777, 0.06626875, 0.066316955, 0.0664092, 0.066376045, 0.0663955, 0.06635159, 0.06640049, 0.06647232, 0.066354126, 0.06650754, 0.06646151, 0.06652975, 0.06644375, 0.066484824, 0.06641694, 0.06634186, 0.06637371, 0.06634477, 0.06623104, 0.066292986, 0.066150635, 0.066064715, 0.06594163, 0.06562378, 0.06501717, 0.064409375, 0.063845314, 0.063771024, 0.06367803, 0.0633516, 0.06284356, 0.062278282, 0.061914273, 0.061428655, 0.060631223, 0.059867263, 0.059564818, 0.059591882, 0.05996957, 0.06021989, 0.06025146, 0.060064588, 0.060311876, 0.060772937, 0.061434347

In [170]:
plt.plot(refs[1:])

### Extract Reflectance

In [12]:
%matplotlib qt

In [3]:
from skimage.morphology import remove_small_objects, remove_small_holes, opening, disk
import spectral.io.envi as envi
import matplotlib.pyplot as plt
from pathlib import Path
import pandas as pd
import numpy as np
import os

In [4]:

data_root = r'C:\Users\baolab\Downloads\biochar_data'

sub_folders = os.listdir(data_root)

print(sub_folders)

['BH7030', 'BH8020', 'BH9010', 'control']


In [5]:
i = 0
data = f'C:/Users/baolab/Downloads/biochar_data\{sub_folders[i]}'

files1 = os.listdir(data)
print(set(files1))
print(len(files1))

file_names_list = []

for i in range(len(files1)):
    
    file_name_1 = files1[i].split('.')[0]
    file_names_list.append(file_name_1)

file_names_list = list(set(file_names_list))

print(file_names_list)
print(len(file_names_list))


{'7030_Oven dried_39_112_2026-02-24_13-01-53_.bin', '7030_Oven dried_1_1_2026-02-24_11-07-50_.bin', '7030_Oven dried_39_112_2026-02-24_13-01-53_.hdr', '7030_Oven dried_34_99_2026-02-24_12-52-25_.hdr', '7030_Oven dried_1_1_2026-02-24_11-07-50_.png', '7030_Oven dried_39_112_2026-02-24_13-01-53_.png', '7030_Oven dried_1_1_2026-02-24_11-07-50_.hdr', '7030_Oven dried_34_99_2026-02-24_12-52-25_.png', '7030_Oven dried_34_99_2026-02-24_12-52-25_.bin'}
9
['7030_Oven dried_39_112_2026-02-24_13-01-53_', '7030_Oven dried_34_99_2026-02-24_12-52-25_', '7030_Oven dried_1_1_2026-02-24_11-07-50_']
3


In [6]:
file_names_list

['7030_Oven dried_39_112_2026-02-24_13-01-53_',
 '7030_Oven dried_34_99_2026-02-24_12-52-25_',
 '7030_Oven dried_1_1_2026-02-24_11-07-50_']

In [7]:

def load_cube(file_name, data_path):
    '''
    Takes in the filename and return its cube
    '''
    data_dir = Path(data_path)

    header_file = str(data_dir / f'{file_name}.hdr')
    spectral_file = str(data_dir /f'{file_name}.bin')

    # open hyperspec cude
    image_cube = envi.open(header_file, spectral_file)

    print(f"Image dimensions (rows, cols, bands): {image_cube.shape}")
    print(f"Number of wavelengths (bands): {image_cube.nbands}")

    return image_cube


In [8]:
data

'C:/Users/baolab/Downloads/biochar_data\\BH7030'

In [9]:
file_name = file_names_list[0]
data_path = data

img_cube1 = load_cube(file_name, data_path)
img_cube1

Image dimensions (rows, cols, bands): (720, 640, 201)
Number of wavelengths (bands): 201


	Data Source:   'C:\Users\baolab\Downloads\biochar_data\BH7030\7030_Oven dried_39_112_2026-02-24_13-01-53_.bin'
	# Rows:            720
	# Samples:         640
	# Bands:           201
	Interleave:        BIP
	Quantization:  32 bits
	Data format:   float32

In [10]:
img_cube1.shape

(720, 640, 201)

In [13]:
band_index = 50

single_band = img_cube1.read_band(band_index)

import matplotlib.pyplot as plt

plt.figure(figsize=(8, 8))

plt.imshow(single_band)#, cmap='gray' # FOR GRAY SCALE IMAGE

plt.show()

In [14]:
# %matplotlib widget

def get_plant_px(img_cube):

    hypercube = img_cube.load()

    display_band = hypercube[:, :, 50]

    plant_coords = [] # List to store plant coordinates

    fig, ax = plt.subplots()

    ax.imshow(display_band, cmap='viridis')
    ax.set_title("Click to select PLANT pixels")

    def onclick_plant(event):

        # Get integer coordinates of the click
        x, y = int(event.xdata), int(event.ydata)

        # Store the coordinates (row, col)
        plant_coords.append([y, x])

        # Draw a marker for visual feedback
        ax.plot(x, y, 'rx', markersize=8)
        fig.canvas.draw()

    # Connect the click event to our function
    cid = fig.canvas.mpl_connect('button_press_event', onclick_plant)
    plt.show() # Display the interactive plot

    return plant_coords

In [15]:
plantsp = get_plant_px(img_cube1)

c:\ProgramData\anaconda3\envs\hyperspec\Lib\site-packages\spectral\io\spyfile.py:224: NaNValueWarning: Image data contains NaN values.
  warnings.warn('Image data contains NaN values.', NaNValueWarning)


In [18]:
print(plantsp)

[[387, 271], [365, 272], [350, 272], [337, 273], [331, 285], [328, 302], [328, 310], [324, 321], [322, 330], [322, 338], [336, 337], [344, 337], [368, 338], [380, 331], [380, 341], [392, 329], [392, 329], [387, 313], [387, 299], [391, 293], [376, 261], [357, 261], [357, 288], [348, 291], [344, 304], [341, 320], [331, 327], [350, 327], [363, 326], [365, 322], [365, 321], [379, 295], [375, 281], [375, 276], [393, 282], [388, 263], [393, 267], [365, 304], [359, 304], [351, 316], [372, 316], [372, 306], [366, 288], [357, 341]]


In [17]:
print(plantsp)

[[387, 271], [365, 272], [350, 272], [337, 273], [331, 285], [328, 302], [328, 310], [324, 321], [322, 330], [322, 338], [336, 337], [344, 337], [368, 338], [380, 331], [380, 341], [392, 329], [392, 329], [387, 313], [387, 299], [391, 293], [376, 261], [357, 261], [357, 288], [348, 291], [344, 304], [341, 320], [331, 327], [350, 327], [363, 326], [365, 322], [365, 321], [379, 295], [375, 281], [375, 276], [393, 282], [388, 263], [393, 267], [365, 304], [359, 304], [351, 316], [372, 316], [372, 306], [366, 288], [357, 341]]


In [19]:

def get_background_px(img_cube):

    hypercube = img_cube.load()
    display_band = hypercube[:, :, 50]

    background_coords = [] # List for background coordinates
    fig2, ax2 = plt.subplots()
    ax2.imshow(display_band, cmap='viridis')
    ax2.set_title("Click to select BACKGROUND pixels")

    def onclick_background(event):
        x, y = int(event.xdata), int(event.ydata)
        background_coords.append([y, x])
        ax2.plot(x, y, 'bo', markersize=5, markerfacecolor='none') # Blue circle marker
        fig2.canvas.draw()

    cid2 = fig2.canvas.mpl_connect('button_press_event', onclick_background)
    plt.show()

    return background_coords


In [20]:
backpx = get_background_px(img_cube1)


c:\ProgramData\anaconda3\envs\hyperspec\Lib\site-packages\spectral\io\spyfile.py:224: NaNValueWarning: Image data contains NaN values.
  warnings.warn('Image data contains NaN values.', NaNValueWarning)


In [22]:
print(backpx)

[[132, 95], [87, 68], [47, 46], [13, 24], [13, 52], [13, 93], [19, 164], [13, 232], [19, 311], [31, 417], [35, 503], [44, 571], [30, 616], [26, 574], [12, 515], [16, 455], [19, 391], [37, 371], [74, 275], [87, 129], [97, 68], [84, 35], [123, 47], [218, 75], [257, 54], [355, 44], [385, 43], [472, 39], [606, 42], [652, 42], [547, 34], [501, 32], [693, 20], [692, 78], [688, 116], [691, 140], [674, 154], [673, 182], [675, 199], [673, 235], [673, 290], [673, 322], [672, 340], [666, 383], [667, 413], [680, 416], [691, 350], [696, 280], [696, 274], [699, 263], [698, 257], [696, 194], [630, 121], [534, 108], [448, 95], [366, 157], [341, 65], [264, 80], [239, 130], [322, 182], [186, 25], [185, 73], [177, 123], [177, 180], [177, 192], [177, 215], [173, 249], [173, 293], [164, 390], [183, 427], [178, 336], [189, 379], [189, 457], [191, 501], [179, 598], [178, 611], [193, 526], [198, 568], [171, 555], [173, 469], [139, 410], [99, 332], [95, 224], [79, 183], [69, 155], [48, 103], [48, 82], [45, 164

In [ ]:
img_cube1.asarray().mac()

In [43]:
img_cube_arr = img_cube1.asarray()
img_cube_arr

memmap([[[0.53088236, 0.475     , 0.48783782, ..., 0.11875   ,
          0.11014493, 0.11026785],
         [0.5029412 , 0.5277778 , 0.53918916, ..., 0.1180791 ,
          0.10935251, 0.11227273],
         [0.5135135 , 0.5277778 , 0.52500004, ..., 0.12344632,
          0.11014493, 0.12954545],
         ...,
         [0.5181818 , 0.58055556, 0.5428571 , ..., 0.1443038 ,
          0.1368    , 0.133     ],
         [0.5588235 , 0.53088236, 0.53088236, ..., 0.14339623,
          0.1281746 , 0.13168317],
         [0.6333333 , 0.57575756, 0.57      , ..., 0.14188312,
          0.1292    , 0.11399999]],

        [[0.53088236, 0.4486111 , 0.53918916, ..., 0.13494319,
          0.11702899, 0.1017857 ],
         [0.5588235 , 0.5013889 , 0.5905405 , ..., 0.12881356,
          0.11618704, 0.12090909],
         [0.5905405 , 0.5277778 , 0.55      , ..., 0.13418078,
          0.11014493, 0.11227273],
         ...,
         [0.5469697 , 0.5277778 , 0.5428571 , ..., 0.13829114,
          0.1292    , 0.1

In [45]:
img_cube_arr[0]

memmap([[0.53088236, 0.475     , 0.48783782, ..., 0.11875   , 0.11014493,
         0.11026785],
        [0.5029412 , 0.5277778 , 0.53918916, ..., 0.1180791 , 0.10935251,
         0.11227273],
        [0.5135135 , 0.5277778 , 0.52500004, ..., 0.12344632, 0.11014493,
         0.12954545],
        ...,
        [0.5181818 , 0.58055556, 0.5428571 , ..., 0.1443038 , 0.1368    ,
         0.133     ],
        [0.5588235 , 0.53088236, 0.53088236, ..., 0.14339623, 0.1281746 ,
         0.13168317],
        [0.6333333 , 0.57575756, 0.57      , ..., 0.14188312, 0.1292    ,
         0.11399999]], dtype=float32)

In [23]:

print(len(backpx))
print(len(plantsp))

259
44


In [13]:

def train_svm(image_cube, plant_coords, background_coords):

    hypercube = image_cube.load()
    # --- Build the training data from your selected points ---
    X_train = []
    y_train = []
    bands = hypercube.shape[2]

    # Add plant pixels (Class 1)
    for r, c in plant_coords:
        X_train.append(hypercube[r, c, :].reshape(bands))
        y_train.append(1)

    # Add background pixels (Class 0)
    for r, c in background_coords:
        X_train.append(hypercube[r, c, :].reshape(bands))
        y_train.append(0)

    X_train = np.array(X_train)
    y_train = np.array(y_train)

    print(f"Training data created successfully!")
    print(f"Plant samples: {len(plant_coords)}")
    print(f"Background samples: {len(background_coords)}")
    print(f"Total training samples: {X_train.shape[0]}")

    # --- Step 2: Train the SVM Classifier ---
    # The 'C' and 'gamma' parameters can be tuned, but defaults are a good start.
    from sklearn.svm import SVC

    classifier = SVC(C=550, gamma='scale')
    classifier.fit(X_train, y_train)
    print("Classifier trained successfully!")

    # --- Step 3: Predict on the Entire Image ---
    # Reshape the hypercube to a long list of pixels (n_pixels, n_bands)
    rows, cols, bands = hypercube.shape
    img_reshaped = hypercube.reshape(rows * cols, bands)

    # Predict the class for every pixel
    predicted_labels = classifier.predict(img_reshaped)

    # Reshape the prediction back into a 2D image mask
    final_mask = predicted_labels.reshape(rows, cols)

    # --- Visualize the Result ---
    plt.figure(figsize=(8, 8))
    plt.imshow(final_mask, cmap='gray')
    plt.title('Segmentation Mask from SVM Classifier')
    plt.show()

    return final_mask, classifier

def predict_svm(svm_classifier, image_cube):

    hypercube = image_cube.load()
    rows, cols, bands = image_cube.shape
    img_reshaped = hypercube.reshape(rows * cols, bands)

    # Predict the class for every pixel
    predicted_labels = svm_classifier.predict(img_reshaped)

    # Reshape the prediction back into a 2D image mask
    final_mask = predicted_labels.reshape(rows, cols)

    return final_mask


In [ ]:
import numpy as np
svm_mask, svm_classifier = train_svm(image_cube, plantsp, backpx)


In [ ]:
svm_classifier

In [ ]:

svm_mask, svm_classifier = train_svm(img_cube1, plantsp, backpx)

In [26]:
# %matplotlib qt
import numpy as np
from pathlib import Path
from utils.utils import run
from utils.train import SVMclassifier
from lima_beans.bean_utils import QtLinePicker, process_and_save_objects

In [ ]:
X, y = run(img_cube1)


In [ ]:

fig, axes = plt.subplots(1, 2, figsize=(12, 6))

axes[0].imshow(svm_mask, cmap='gray')
axes[0].set_title('SVM')

axes[1].imshow(single_band, cmap='gray')
axes[1].set_title('Final Mask New')


### X

In [1]:
%matplotlib qt
import numpy as np
from pathlib import Path
from utils.utils import run
from utils.train import SVMclassifier
from lima_beans.bean_utils import QtLinePicker, process_and_save_objects

In [3]:
from biochar.biochar_utils import extract_reflectance_cube


In [ ]:
extract_reflectance_cube

import numpy as np
import os

def save_reflectance_np(filepath, ref):
    np.save(filepath, ref)
    

In [ ]:
data_root = 'C:/Users/baolab/Documents/Cubert/2025_09_03_11-12-59'
file_names = ['3_000_0012', '10_000_0010', '36_000_0008', '41_000_0014', '51_000_0008']

dest_root = 'data/numpy_files'
os.makedirs(dest_root, exist_ok=True)

for filename in file_names:

    cuvis_session_file = f'{data_root}/{filename}.cu3s'
    # print(cuvis_session_file)
    reflectance = extract_reflectance_cube(cuvis_session_file)

    dest_path = f'{dest_root}/{filename}.npy'
    # print(dest_path)
    save_reflectance_np(dest_path, reflectance)
